In [9]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import json
from peft import prepare_model_for_kbit_training

In [10]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [11]:
MODEL_NAME = "Qwen/Qwen3-8B"


In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [13]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading checkpoint shards: 100%|██████████| 5/5 [01:51<00:00, 22.30s/it]


In [14]:
dataset = load_dataset(
    "json",
    data_files= "./dataset/v3a_corrected.jsonl",
)

Generating train split: 40212 examples [00:03, 10372.96 examples/s]


DatasetGenerationError: An error occurred while generating the dataset

In [25]:
import json

INPUT_FILE = "V5A_nested.jsonl"
OUTPUT_FILE = "V5A_clean.jsonl"

total = 0
fixed = 0
errors = 0

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line_num, line in enumerate(fin, 1):

        if not line.strip():
            continue

        try:
            data = json.loads(line)

            # -------------------------
            # INSTRUCTION
            # -------------------------

            instruction = data.get("instruction", "")

            if not isinstance(instruction, str):
                instruction = str(instruction)
                fixed += 1

            # -------------------------
            # INPUT
            # -------------------------

            input_data = data.get("input", {})

            # If input is JSON string, decode it
            if isinstance(input_data, str):
                try:
                    input_data = json.loads(input_data)
                    fixed += 1
                except:
                    input_data = {}

            if not isinstance(input_data, dict):
                input_data = {}
                fixed += 1

            domain = input_data.get("domain", "")
            relationship = input_data.get("relationship", "")
            conversation = input_data.get("conversation", "")

            # Force everything to string
            if not isinstance(domain, str):
                domain = str(domain)

            if not isinstance(relationship, str):
                relationship = str(relationship)

            if not isinstance(conversation, str):
                conversation = str(conversation)

            # -------------------------
            # OUTPUT
            # -------------------------

            output_data = data.get("output", {})

            # If output is JSON string, decode it
            if isinstance(output_data, str):
                try:
                    output_data = json.loads(output_data)
                    fixed += 1
                except:
                    output_data = {}

            if not isinstance(output_data, dict):
                output_data = {}
                fixed += 1

            primary = output_data.get("primary_objective", "")
            secondary = output_data.get("secondary_objective", "")
            priority = output_data.get("priority", "")
            reason = output_data.get("reason", "")

            # Force everything to string
            if not isinstance(primary, str):
                primary = str(primary)

            if not isinstance(secondary, str):
                secondary = str(secondary)

            if not isinstance(priority, str):
                priority = str(priority)

            if not isinstance(reason, str):
                reason = str(reason)

            # -------------------------
            # REBUILD EXACT SCHEMA
            # -------------------------

            clean_data = {
                "instruction": instruction,
                "input": {
                    "domain": domain,
                    "relationship": relationship,
                    "conversation": conversation
                },
                "output": {
                    "primary_objective": primary,
                    "secondary_objective": secondary,
                    "priority": priority,
                    "reason": reason
                }
            }

            fout.write(
                json.dumps(
                    clean_data,
                    ensure_ascii=False
                ) + "\n"
            )

            total += 1

        except Exception as e:
            errors += 1
            print(f"Line {line_num}: {e}")

print("=" * 60)
print("V5A CLEANING COMPLETE")
print("=" * 60)
print("Total samples :", total)
print("Fixed records :", fixed)
print("Errors        :", errors)
print("Output file   :", OUTPUT_FILE)

V5A CLEANING COMPLETE
Total samples : 104036
Fixed records : 0
Errors        : 0
Output file   : V5A_clean.jsonl


In [15]:
import json

# ==========================================================
# Prepare Model
# ==========================================================

model = prepare_model_for_kbit_training(model)

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.config.use_cache = False

# ==========================================================
# System Prompt
# ==========================================================

SYSTEM_PROMPT = """You are an expert assistant objective prediction model.

Your task is to predict the assistant's objective from the given conversation.

Rules:
- Predict only the assistant's objective.
- Do NOT generate a reply.
- Do NOT summarize the conversation.
- Do NOT extract memories.
- Do NOT infer unsupported information.
- Base your prediction only on the provided conversation.

Return ONLY valid JSON in the following format:

{
  "primary_objective": "",
  "secondary_objective": "",
  "priority": "",
  "reason": ""
}

Field Definitions:
- primary_objective: The assistant's main objective.
- secondary_objective: An optional supporting objective. Use "None" if no meaningful secondary objective exists.
- priority: One of "High", "Medium", or "Low".
- reason: A brief explanation (1–2 sentences) describing why these objectives were selected.

Return only the JSON object. Do not include any extra text or markdown.
"""

# ==========================================================
# Formatting Function
# ==========================================================

def formatting_func(example):

    user_input = (
        f"{example['instruction']}\n\n"
        f"{json.dumps(example['input'], ensure_ascii=False, separators=(',', ':'))}"
    )

    assistant_output = json.dumps(
        example["output"],
        ensure_ascii=False,
        separators=(",", ":")
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_input,
        },
        {
            "role": "assistant",
            "content": assistant_output,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

In [18]:
# Printing  TOKEN ,DISTRIBUTION ,FORMATTING ,LONGEST ,SAMPLES of this dataset (important thing before training)

import time
from statistics import mean, median

def analyze_dataset(name, dataset, tokenizer, formatting_func):
    print("\n" + "="*70)
    print(name)
    print("="*70)

    train = dataset["train"]

    lengths = []
    format_times = []

    start_total = time.time()

    for i, sample in enumerate(train):
        t1 = time.time()

        text = formatting_func(sample)

        t2 = time.time()
        format_times.append(t2 - t1)

        tokens = tokenizer(text, add_special_tokens=True)["input_ids"]
        lengths.append(len(tokens))

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{len(train)}")

    total_time = time.time() - start_total

    print("\n----- BASIC -----")
    print("Samples              :", len(train))
    print("Average Tokens       :", round(mean(lengths),2))
    print("Median Tokens        :", median(lengths))
    print("Minimum Tokens       :", min(lengths))
    print("Maximum Tokens       :", max(lengths))

    print("\n----- TOKEN DISTRIBUTION -----")
    print(">256 tokens          :", sum(x > 256 for x in lengths))
    print(">512 tokens          :", sum(x > 512 for x in lengths))
    print(">1024 tokens         :", sum(x > 1024 for x in lengths))
    print(">2048 tokens         :", sum(x > 2048 for x in lengths))
    print(">4096 tokens         :", sum(x > 4096 for x in lengths))

    print("\n----- FORMATTING -----")
    print("Formatting Time      :", round(total_time,2), "sec")
    print("Average/sample       :", round(mean(format_times)*1000,3), "ms")
    print("Samples/sec          :", round(len(train)/total_time,2))

    print("\n----- LONGEST SAMPLES -----")
    top = sorted(enumerate(lengths), key=lambda x: x[1], reverse=True)[:10]

    for idx, tok in top:
        print(f"Sample {idx:6d} : {tok} tokens")

    return lengths


In [19]:
old_dataset = load_dataset(
    "json",
    data_files= r"./v5a/V5A.jsonl",
  
)

# new_dataset = load_dataset(
#     "json",
#     data_files=r"./newV4a/finalV4A.jsonl",
  
# )


old_lengths = analyze_dataset(
    "OLD DATASET",
    old_dataset,
    tokenizer,
    formatting_func
)

# new_lengths = analyze_dataset(
#     "NEW V4A DATASET",
#     new_dataset,
#     tokenizer,
#     formatting_func
# )


OLD DATASET
Processed 5000/59921
Processed 10000/59921
Processed 15000/59921
Processed 20000/59921
Processed 25000/59921
Processed 30000/59921
Processed 35000/59921
Processed 40000/59921
Processed 45000/59921
Processed 50000/59921
Processed 55000/59921

----- BASIC -----
Samples              : 59921
Average Tokens       : 446.63
Median Tokens        : 439
Minimum Tokens       : 303
Maximum Tokens       : 879

----- TOKEN DISTRIBUTION -----
>256 tokens          : 59921
>512 tokens          : 10744
>1024 tokens         : 0
>2048 tokens         : 0
>4096 tokens         : 0

----- FORMATTING -----
Formatting Time      : 87.0 sec
Average/sample       : 0.23 ms
Samples/sec          : 688.73

----- LONGEST SAMPLES -----
Sample  49784 : 879 tokens
Sample  24899 : 877 tokens
Sample  54440 : 850 tokens
Sample  26312 : 848 tokens
Sample  17506 : 842 tokens
Sample   9744 : 836 tokens
Sample  17119 : 833 tokens
Sample  39085 : 819 tokens
Sample  48582 : 818 tokens
Sample   4658 : 808 tokens
